 Como consultar dados com SQL


Um caso real de Compras Públicas Federais

A demanda chegou assim:

**“Precisamos entender onde o valor licitado se concentra e quais estruturas públicas aparecem nesses registros. Você consegue investigar?”**

**Vamos usar SQL para aprender a:**

consultar dados com SELECT;

filtrar com WHERE;

ordenar com ORDER BY;

resumir com COUNT(), SUM() e GROUP BY;

conectar tabelas com JOIN;

levar um resultado SQL para o pandas.

In [2]:
import base64
import gzip
import sqlite3
import pandas as pd
from pathlib import Path
from IPython.display import display

PASTA_TRABALHO = Path("/content") if Path("/content").exists() else Path.cwd()
ARQUIVO_BANCO = PASTA_TRABALHO / "compras_publicas_2018.sqlite"

# O banco real da aula está compactado dentro deste notebook para que
# você precise de apenas UM arquivo para estudar depois.

In [3]:
#@title 3. Banco compactado — execute esta célula (não precisa abrir)
# O banco real da aula está armazenado aqui em formato compactado.
# Esta célula fica fechada porque o conteúdo é muito grande e não é parte da explicação.
BANCO_COMPACTADO_B64 = """..."""


In [4]:
# 4. Preparação — carregando o banco e criando a função SQL
if not ARQUIVO_BANCO.exists():
    conteudo = gzip.decompress(base64.b64decode(BANCO_COMPACTADO_B64))
    ARQUIVO_BANCO.write_bytes(conteudo)

conexao = sqlite3.connect(ARQUIVO_BANCO)

# Função auxiliar: recebe uma query SQL e mostra o resultado como tabela.
def sql(query):
    return pd.read_sql_query(query, conexao)

print("✅ Banco carregado!")
print(f"Arquivo: {ARQUIVO_BANCO}")


✅ Banco carregado!
Arquivo: /content/compras_publicas_2018.sqlite


Entendendo o banco antes de consultar

SELECT name
FROM sqlite_master
WHERE type = 'table'; - tipo de consulta

In [5]:
sql("""
SELECT *
FROM sqlite_master
""")

,type,name,tbl_name,rootpage,sql
0,table,orgaos_superiores,orgaos_superiores,2,CREATE TABLE orgaos_superiores (\n codigo_org...
1,index,sqlite_autoindex_orgaos_superiores_1,orgaos_superiores,3,None
2,table,orgaos,orgaos,4,CREATE TABLE orgaos (\n codigo_orgao TEXT PRI...
3,index,sqlite_autoindex_orgaos_1,orgaos,5,None
4,table,unidades_gestoras,unidades_gestoras,6,CREATE TABLE unidades_gestoras (\n codigo_ug ...
5,index,sqlite_autoindex_unidades_gestoras_1,unidades_gestoras,7,None
6,table,modalidades,modalidades,8,CREATE TABLE modalidades (\n codigo_modalidad...
7,index,sqlite_autoindex_modalidades_1,modalidades,9,None
8,table,licitacoes,licitacoes,10,CREATE TABLE licitacoes (\n id_licitacao INTE...
9,table,sqlite_sequence,sqlite_sequence,11,"CREATE TABLE sqlite_sequence(name,seq)"


In [6]:
sql("""
SELECT name
FROM sqlite_master
WHERE type = 'table'
  AND name NOT LIKE 'sqlite_%'
ORDER BY name;
""")

,name
0,licitacoes
1,modalidades
2,orgaos
3,orgaos_superiores
4,unidades_gestoras


A pergunta mais simples possível é:

**“Quero olhar algumas licitações.”**

In [7]:
sql("""
SELECT *
FROM licitacoes
LIMIT 2;
""")

,id_licitacao,numero_licitacao,codigo_ug,codigo_modalidade,numero_processo,objeto,situacao,uf,municipio,data_resultado,data_abertura,valor_licitacao
0,1,000012016,160438,-99,80871182401201629,Objeto: Pregão Eletrônico - Aquisição de mater...,Evento de Resultado de Julgame,RS,URUGUAIANA,2018-12-04,2016-08-17,18854.00
1,2,000012017,160203,3,64040053530201713,Objeto: Contratação de empresa para construção...,Evento de Resultado de Julgame,PI,TERESINA,2018-12-07,None,889571.43


In [8]:
sql("""
SELECT
    numero_licitacao,
    objeto,
    uf,
    municipio,
    valor_licitacao
FROM licitacoes
LIMIT 10;
""")


,numero_licitacao,objeto,uf,municipio,valor_licitacao
0,000012016,Objeto: Pregão Eletrônico - Aquisição de mater...,RS,URUGUAIANA,18854.00
1,000012017,Objeto: Contratação de empresa para construção...,PI,TERESINA,889571.43
2,000012017,Objeto: Contratação de empresa especializada n...,CE,FORTALEZA,865536.65
3,000012018,"Objeto: TV LED de 65 polegadas, UHD 4k, com co...",-3,Inválido,11075.99
4,000012018,Objeto: Pretação de serviços de publicação de ...,DF,BRASILIA,12000.00
5,000012018,Objeto: Licença de uso irrestrito da cópia do ...,RJ,RIO DE JANEIRO,1590.00
6,000012018,Objeto: Contratação de empresa especializada n...,DF,BRASILIA,2677075.97
7,000012018,Objeto: Pregão Eletrônico - Contratação exclus...,GO,ABADIA DE GOIAS,124000.00
8,000012018,Objeto: Contratação de empresa especializada p...,DF,BRASILIA,1692588.08
9,000012018,Objeto: Construção do posto de combustível e l...,PA,BELEM,194247.89


Qual parte da consulta responde cada pergunta?

Quais colunas quero? → **SELECT**

Em qual tabela? → **FROM**

Quais linhas entram? → **WHERE**

Quantas linhas quero visualizar agora? → **LIMIT**

WHERE: ficando só com o que interessa
Agora a pergunta muda:

**“Quero olhar apenas licitações associadas a São Paulo.”**

In [9]:
sql("""
SELECT
    numero_licitacao,
    municipio,
    objeto,
    valor_licitacao
FROM licitacoes
WHERE uf = 'MG'
LIMIT 10;
""")

,numero_licitacao,municipio,objeto,valor_licitacao
0,000012018,POUSO ALEGRE,Objeto: Pregão Eletrônico - Contratação de emp...,0.00
1,000012018,BELO HORIZONTE,"Objeto: Licitação na modalidade CONCORRÊNCIA, ...",3838471.11
2,000012018,BARBACENA,Objeto: Pregão Eletrônico - Contratação de Emp...,13998.91
3,000012018,TEOFILO OTONI,Objeto: Trata-se de contratação de serviços de...,8400.00
4,000022018,BELO HORIZONTE,Objeto: Contratação de empresa prestadora de s...,50000.00
5,000022018,BELO HORIZONTE,Objeto: Serviços de água e esgoto para a Sede ...,265989.17
6,000022018,OURO PRETO,Objeto: Contratação de empresa especializada n...,1095142.52
7,000022018,OURO PRETO,Objeto: Contratação de empresa especializada n...,1871169.37
8,000022018,BETIM,Objeto: Pregão Eletrônico - Aquisição de Mater...,1269.60
9,000022018,TEOFILO OTONI,Objeto: Contratação de empresa especializada n...,220078.03


ORDER BY: **“quero ver os maiores primeiro”**

A nova pergunta é:

**“Entre as licitações de SP, quais têm os maiores valores registrados?”**

Precisamos ordenar o resultado.

ORDER BY valor_licitacao DESC

ASC → crescente

DESC → decrescente

In [10]:
sql("""
SELECT
    numero_licitacao,
    municipio,
    objeto,
    valor_licitacao
FROM licitacoes
WHERE uf = 'SP' AND valor_licitacao > 50
ORDER BY valor_licitacao ASC
LIMIT 10;
""")


,numero_licitacao,municipio,objeto,valor_licitacao
0,010382018,SAO PAULO,Objeto: Aquisição de Ferramentas,52.00
1,002752018,SAO CARLOS,Objeto: Aquisição de material eletrônico (ardu...,54.25
2,001982018,SAO CARLOS,Objeto: Colarinho em chapa galvanizada,64.00
3,002682018,SAO CARLOS,Objeto: Material de Expediente.,67.85
4,000092018,GUARULHOS,Objeto: Contratação de pessoa jurídica para fo...,70.00
5,002692018,SAO CARLOS,Objeto: Material de Expediente.,91.00
6,002712018,SAO CARLOS,Objeto: Material de Expediente,91.94
7,000192018,CAMPINAS,Objeto: Contratação de empresa especializada e...,95.00
8,001512018,SAO PAULO,Objeto: Contratação de empresa especializada n...,100.00
9,002302018,SAO CARLOS,Objeto: Aquisição de material de combate a inc...,105.00


Agregações: quando não queremos olhar linha por linha
Nem sempre a pergunta é sobre uma licitação específica.

Às vezes queremos resumir o conjunto.

Algumas funções muito comuns:

função	pergunta que ajuda a responder

**COUNT(*)** quantas linhas existem?

**SUM(coluna)**	qual é a soma?

**AVG(coluna)**	qual é a média?

**MIN(coluna)**	qual é o menor valor?

**MAX(coluna)**	qual é o maior valor?


In [11]:
sql("""
SELECT COUNT(*)
FROM licitacoes;
""")

,COUNT(*)
0,15336


In [12]:
sql("""
SELECT COUNT(*) AS quantidade_licitacoes
FROM licitacoes;
""")


,quantidade_licitacoes
0,15336


In [13]:
sql("""
SELECT AVG(valor_licitacao) AS média_licitacoes
FROM licitacoes;
""")

,média_licitacoes
0,580598.062503


O que é AS?
AS cria um apelido (alias) para a coluna no resultado.

Sem alias, poderíamos receber uma coluna chamada:

COUNT(*)
Com:

COUNT(*) AS quantidade_licitacoes

**GROUP BY - **
Agora surge uma pergunta realmente analítica:

**“Quais UFs concentram mais valor licitado no snapshot?”**

In [14]:
sql("""
SELECT
    uf,
    COUNT(*) AS quantidade_licitacoes,
    SUM(valor_licitacao) AS valor_total_licitado
FROM licitacoes
WHERE uf IS NOT NULL
  AND TRIM(uf) <> ''
GROUP BY uf
ORDER BY valor_total_licitado DESC
LIMIT 10;
""")


,uf,quantidade_licitacoes,valor_total_licitado
0,DF,1686,4.762520e+09
1,RJ,2977,1.517958e+09
2,RS,1170,4.650272e+08
3,SP,1203,3.828371e+08
4,MG,1305,3.349053e+08
5,PE,595,2.307002e+08
6,PB,266,1.281990e+08
7,BA,432,1.017522e+08
8,PR,759,9.606969e+07
9,SC,479,9.396330e+07


*Checkpoint rápido*
Até aqui você já viu que:

**SELECT** escolhe colunas;

**FROM** aponta a tabela;

**WHERE** filtra linhas;

**ORDER BY** organiza o resultado;

**GROUP BY** cria grupos para resumir o conjunto.

O problema que cria a necessidade de JOIN
Agora quero responder:

**“Quais modalidades de licitação aparecem mais?”**

Vamos olhar as colunas da tabela licitacoes.

In [15]:
sql("""
PRAGMA table_info(licitacoes);
""")


,cid,name,type,notnull,dflt_value,pk
0,0,id_licitacao,INTEGER,0,None,1
1,1,numero_licitacao,TEXT,1,None,0
2,2,codigo_ug,TEXT,1,None,0
3,3,codigo_modalidade,TEXT,0,None,0
4,4,numero_processo,TEXT,0,None,0
5,5,objeto,TEXT,0,None,0
6,6,situacao,TEXT,0,None,0
7,7,uf,TEXT,0,None,0
8,8,municipio,TEXT,0,None,0
9,9,data_resultado,TEXT,0,None,0


**JOIN**: conectando as peças
Estrutura mental:

FROM tabela_A a

JOIN tabela_B b

  ON a.chave = b.chave

Leia como:

**“Pegue a tabela A e conecte com a tabela B quando estas chaves forem iguais.”**

In [16]:
sql("""
SELECT *
FROM licitacoes
JOIN modalidades
    ON licitacoes.codigo_modalidade = modalidades.codigo_modalidade;
""")

,id_licitacao,numero_licitacao,codigo_ug,codigo_modalidade,numero_processo,objeto,situacao,uf,municipio,data_resultado,data_abertura,valor_licitacao,codigo_modalidade,modalidade
0,1,000012016,160438,-99,80871182401201629,Objeto: Pregão Eletrônico - Aquisição de mater...,Evento de Resultado de Julgame,RS,URUGUAIANA,2018-12-04,2016-08-17,18854.00,-99,Pregão - Registro de Preço
1,2,000012017,160203,3,64040053530201713,Objeto: Contratação de empresa para construção...,Evento de Resultado de Julgame,PI,TERESINA,2018-12-07,None,889571.43,3,Concorrência
2,3,000012017,193002,2,59400003079201636,Objeto: Contratação de empresa especializada n...,Evento de Resultado de Julgame,CE,FORTALEZA,2018-12-07,None,865536.65,2,Tomada de Preços
3,4,000012018,925135,6,148097,"Objeto: TV LED de 65 polegadas, UHD 4k, com co...",Encerrado,-3,Inválido,2018-12-04,2018-11-30,11075.99,6,Dispensa de Licitação
4,5,000012018,110101,7,00300000433201818,Objeto: Pretação de serviços de publicação de ...,Publicado,DF,BRASILIA,2018-12-21,None,12000.00,7,Inexigibilidade de Licitação
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15331,15332,920182018,135023,-99,21150000453201820,Objeto: Pregão Eletrônico - Contratar Empresa ...,Evento de Resultado de Julgame,RJ,SEROPEDICA,2018-12-04,2018-11-26,2588060.08,-99,Pregão - Registro de Preço
15332,15333,924192017,158274,6,23318002419201784,Objeto: Retificação da dispensa de licitação n...,Publicado,RJ,CAMPOS DOS GOYTACAZES,2018-12-26,None,490325.00,6,Dispensa de Licitação
15333,15334,990922018,158139,6,23317009092201862,Objeto: Aquisição de material para revisão do ...,Encerrado,RJ,CAMPOS DOS GOYTACAZES,2018-12-03,None,1149.75,6,Dispensa de Licitação
15334,15335,996252018,153035,6,23085000625201811,Objeto: Pagamento da taxa referente ao primeir...,Encerrado,MG,UBERABA,2018-12-04,None,298.00,6,Dispensa de Licitação


In [17]:
sql("""
SELECT
    modalidades.modalidade,
    licitacoes.valor_licitacao
FROM licitacoes
JOIN modalidades
    ON licitacoes.codigo_modalidade = modalidades.codigo_modalidade;
""")

,modalidade,valor_licitacao
0,Pregão - Registro de Preço,18854.00
1,Concorrência,889571.43
2,Tomada de Preços,865536.65
3,Dispensa de Licitação,11075.99
4,Inexigibilidade de Licitação,12000.00
...,...,...
15331,Pregão - Registro de Preço,2588060.08
15332,Dispensa de Licitação,490325.00
15333,Dispensa de Licitação,1149.75
15334,Dispensa de Licitação,298.00


In [18]:
sql("""
SELECT
    m.modalidade,
    l.valor_licitacao
FROM licitacoes AS l
JOIN modalidades AS m
    ON l.codigo_modalidade = m.codigo_modalidade;
""")

,modalidade,valor_licitacao
0,Pregão - Registro de Preço,18854.00
1,Concorrência,889571.43
2,Tomada de Preços,865536.65
3,Dispensa de Licitação,11075.99
4,Inexigibilidade de Licitação,12000.00
...,...,...
15331,Pregão - Registro de Preço,2588060.08
15332,Dispensa de Licitação,490325.00
15333,Dispensa de Licitação,1149.75
15334,Dispensa de Licitação,298.00


In [19]:
sql("""
SELECT
    m.modalidade,
    COUNT(*)
FROM licitacoes AS l
JOIN modalidades AS m
    ON l.codigo_modalidade = m.codigo_modalidade
GROUP BY m.modalidade;
""")

,modalidade,COUNT(*)
0,Concorrência,76
1,Concorrência - Registro de Preço,2
2,Concorrência Internacional,3
3,Concurso,4
4,Convite,29
5,Dispensa de Licitação,9831
6,Inexigibilidade de Licitação,1681
7,Pregão,1794
8,Pregão - Registro de Preço,1739
9,Tomada de Preços,177


In [20]:
sql("""
SELECT
    m.modalidade,
    COUNT(*) AS quantidade_licitacoes,
    SUM(l.valor_licitacao) AS valor_total_licitado
FROM licitacoes AS l
JOIN modalidades AS m
    ON l.codigo_modalidade = m.codigo_modalidade
GROUP BY m.modalidade
ORDER BY quantidade_licitacoes DESC;
""")

,modalidade,quantidade_licitacoes,valor_total_licitado
0,Dispensa de Licitação,9831,2.954338e+09
1,Pregão,1794,2.049593e+09
2,Pregão - Registro de Preço,1739,5.825939e+08
3,Inexigibilidade de Licitação,1681,2.842681e+09
4,Tomada de Preços,177,1.196474e+08
5,Concorrência,76,2.543660e+08
6,Convite,29,3.676188e+06
7,Concurso,4,5.730000e+04
8,Concorrência Internacional,3,9.709907e+07
9,Concorrência - Registro de Preço,2,0.000000e+00


 **INNER JOIN** x ***LEFT JOIN***
O **JOIN** que usamos sem especificar o tipo é, por padrão, um **INNER JOIN**.


**INNER JOIN**

Mantém somente linhas que encontram correspondência dos dois lados.


A ── combinação encontrada ── B   ✅ permanece

**LEFT JOIN**

Mantém todas as linhas da tabela da esquerda, mesmo quando não existe correspondência à direita.


A ── sem correspondência ── ✅ permanece             NULL

Para a aula de hoje, o mais importante é entender **a pergunta que cada JOIN responde**, não decorar todos os tipos possíveis.

O que são **l** e **m**?
São aliases de tabela.

licitacoes AS **l**
modalidades AS **m**
Depois disso:

l.codigo_modalidade
m.modalidade

O que são **l** e **m**?

São aliases de tabela.

licitacoes AS **l**

modalidades AS **m**

Depois disso:


l.codigo_modalidade
m.modalidade

In [21]:
sql("""
SELECT COUNT(*) AS quantidade
FROM licitacoes AS l
JOIN modalidades AS m
    ON l.codigo_modalidade = m.codigo_modalidade;
""")

,quantidade
0,15336


**JOIN** em cadeia: o “boss fight” da aula 😈

A pergunta agora é:


**“Quais órgãos superiores concentram maior valor licitado no snapshot?”**


O problema: licitacoes não conhece diretamente orgaos_superiores.

In [22]:
sql("""
SELECT
    os.nome_orgao_superior,
    COUNT(*) AS quantidade_licitacoes,
    SUM(l.valor_licitacao) AS valor_total_licitado
FROM licitacoes AS l
JOIN unidades_gestoras AS ug
    ON l.codigo_ug = ug.codigo_ug
JOIN orgaos AS o
    ON ug.codigo_orgao = o.codigo_orgao
JOIN orgaos_superiores AS os
    ON o.codigo_orgao_superior = os.codigo_orgao_superior
GROUP BY os.nome_orgao_superior
ORDER BY valor_total_licitado DESC
LIMIT 10;
""")


,nome_orgao_superior,quantidade_licitacoes,valor_total_licitado
0,Ministério da Saúde,1173,3.502379e+09
1,Ministério da Educação,4391,1.509284e+09
2,Ministério da Defesa,3806,1.027182e+09
3,Presidência da República,119,7.118403e+08
4,Ministério da Justiça e Segurança Pública,313,4.224208e+08
5,Ministério dos Transportes,188,3.295274e+08
6,Sem informação,193,2.136897e+08
7,Ministério da Gestão e da Inovação em Ser,369,1.882696e+08
8,"Ministério da Ciência, Tecnologia e Inovaç",859,1.812521e+08
9,Ministério da Fazenda,489,1.672926e+08


*Não leia essa query como um bloco assustador*

Leia em camadas:

**Comece em** licitacoes.

Descubra a **unidade gestora** pelo codigo_ug.

Descubra o **órgão** pelo codigo_orgao.

Descubra o **órgão superior** pelo codigo_orgao_superior.

Agora que temos o nome do órgão superior, agrupamos por ele.

Somamos valor_licitacao dentro de cada grupo.

Ordenamos do maior para o menor.

Essa é uma habilidade muito importante em SQL:


**antes de escrever a consulta, desenhe o caminho entre as tabelas.**

Uma **query** completa, do problema ao resultado
Pergunta:

**“Quais são as 10 licitações de maior valor do snapshot e qual unidade gestora e modalidade aparecem em cada uma?”**

In [23]:
licitacoes_df = sql("""
SELECT *
FROM licitacoes
""")

In [24]:
licitacoes_df

,id_licitacao,numero_licitacao,codigo_ug,codigo_modalidade,numero_processo,objeto,situacao,uf,municipio,data_resultado,data_abertura,valor_licitacao
0,1,000012016,160438,-99,80871182401201629,Objeto: Pregão Eletrônico - Aquisição de mater...,Evento de Resultado de Julgame,RS,URUGUAIANA,2018-12-04,2016-08-17,18854.00
1,2,000012017,160203,3,64040053530201713,Objeto: Contratação de empresa para construção...,Evento de Resultado de Julgame,PI,TERESINA,2018-12-07,None,889571.43
2,3,000012017,193002,2,59400003079201636,Objeto: Contratação de empresa especializada n...,Evento de Resultado de Julgame,CE,FORTALEZA,2018-12-07,None,865536.65
3,4,000012018,925135,6,148097,"Objeto: TV LED de 65 polegadas, UHD 4k, com co...",Encerrado,-3,Inválido,2018-12-04,2018-11-30,11075.99
4,5,000012018,110101,7,00300000433201818,Objeto: Pretação de serviços de publicação de ...,Publicado,DF,BRASILIA,2018-12-21,None,12000.00
...,...,...,...,...,...,...,...,...,...,...,...,...
15331,15332,920182018,135023,-99,21150000453201820,Objeto: Pregão Eletrônico - Contratar Empresa ...,Evento de Resultado de Julgame,RJ,SEROPEDICA,2018-12-04,2018-11-26,2588060.08
15332,15333,924192017,158274,6,23318002419201784,Objeto: Retificação da dispensa de licitação n...,Publicado,RJ,CAMPOS DOS GOYTACAZES,2018-12-26,None,490325.00
15333,15334,990922018,158139,6,23317009092201862,Objeto: Aquisição de material para revisão do ...,Encerrado,RJ,CAMPOS DOS GOYTACAZES,2018-12-03,None,1149.75
15334,15335,996252018,153035,6,23085000625201811,Objeto: Pagamento da taxa referente ao primeir...,Encerrado,MG,UBERABA,2018-12-04,None,298.00


In [25]:
top_10 = sql("""
SELECT
    l.numero_licitacao,
    ug.nome_ug,
    m.modalidade,
    l.uf,
    l.municipio,
    ROUND(l.valor_licitacao, 2) AS valor_licitacao
FROM licitacoes AS l
JOIN unidades_gestoras AS ug
    ON l.codigo_ug = ug.codigo_ug
JOIN modalidades AS m
    ON l.codigo_modalidade = m.codigo_modalidade
ORDER BY l.valor_licitacao DESC
LIMIT 10;
""")

top_10

,numero_licitacao,nome_ug,modalidade,uf,municipio,valor_licitacao
0,006372018,DEPARTAMENTO DE LOGISTICA EM SAUDE - DLOG,Dispensa de Licitação,DF,BRASILIA,9.689600e+08
1,000532018,DEPARTAMENTO DE LOGISTICA EM SAUDE - DLOG,Inexigibilidade de Licitação,DF,BRASILIA,5.987247e+08
2,002872018,UNIVERSIDADE FEDERAL DO RIO GRANDE DO SUL,Dispensa de Licitação,RS,PORTO ALEGRE,3.570000e+08
3,000552018,DEPARTAMENTO DE LOGISTICA EM SAUDE - DLOG,Inexigibilidade de Licitação,DF,BRASILIA,3.283033e+08
4,000562018,DEPARTAMENTO DE LOGISTICA EM SAUDE - DLOG,Inexigibilidade de Licitação,DF,BRASILIA,3.069056e+08
5,000102018,INTERVENCAO FEDERAL RIO DE JANEIRO,Pregão,DF,BRASILIA,2.416179e+08
6,000062018,SUPERINTENDENCIA DA PRF EM SAO PAULO,Pregão - Registro de Preço,SP,SAO PAULO,1.430040e+08
7,000492018,DEPARTAMENTO DE LOGISTICA EM SAUDE - DLOG,Inexigibilidade de Licitação,DF,BRASILIA,1.424724e+08
8,000402018,INTERVENCAO FEDERAL RIO DE JANEIRO,Pregão,DF,BRASILIA,1.373064e+08
9,000422018,CENTRO DE AQUISICOES ESPECIFICAS,Inexigibilidade de Licitação,RJ,RIO DE JANEIRO,1.273905e+08


In [26]:
top_10[top_10['uf']=="DF"]

,numero_licitacao,nome_ug,modalidade,uf,municipio,valor_licitacao
0,006372018,DEPARTAMENTO DE LOGISTICA EM SAUDE - DLOG,Dispensa de Licitação,DF,BRASILIA,9.689600e+08
1,000532018,DEPARTAMENTO DE LOGISTICA EM SAUDE - DLOG,Inexigibilidade de Licitação,DF,BRASILIA,5.987247e+08
3,000552018,DEPARTAMENTO DE LOGISTICA EM SAUDE - DLOG,Inexigibilidade de Licitação,DF,BRASILIA,3.283033e+08
4,000562018,DEPARTAMENTO DE LOGISTICA EM SAUDE - DLOG,Inexigibilidade de Licitação,DF,BRASILIA,3.069056e+08
5,000102018,INTERVENCAO FEDERAL RIO DE JANEIRO,Pregão,DF,BRASILIA,2.416179e+08
7,000492018,DEPARTAMENTO DE LOGISTICA EM SAUDE - DLOG,Inexigibilidade de Licitação,DF,BRASILIA,1.424724e+08
8,000402018,INTERVENCAO FEDERAL RIO DE JANEIRO,Pregão,DF,BRASILIA,1.373064e+08


** SQL + pandas**: não precisamos escolher um “time”

In [27]:
consulta = """
SELECT
    uf,
    COUNT(*) AS quantidade_licitacoes,
    SUM(valor_licitacao) AS valor_total_licitado
FROM licitacoes
WHERE uf IS NOT NULL AND TRIM(uf) <> ''
GROUP BY uf
ORDER BY valor_total_licitado DESC;
"""

resultado_uf = pd.read_sql_query(consulta, conexao)
resultado_uf.head()


,uf,quantidade_licitacoes,valor_total_licitado
0,DF,1686,4.762520e+09
1,RJ,2977,1.517958e+09
2,RS,1170,4.650272e+08
3,SP,1203,3.828371e+08
4,MG,1305,3.349053e+08


In [28]:
# Agora o resultado é um DataFrame normal do pandas.
resultado_uf["valor_medio_por_licitacao"] = (
    resultado_uf["valor_total_licitado"]
    / resultado_uf["quantidade_licitacoes"]
)

resultado_uf.head()


,uf,quantidade_licitacoes,valor_total_licitado,valor_medio_por_licitacao
0,DF,1686,4.762520e+09,2.824745e+06
1,RJ,2977,1.517958e+09,5.098952e+05
2,RS,1170,4.650272e+08,3.974591e+05
3,SP,1203,3.828371e+08,3.182354e+05
4,MG,1305,3.349053e+08,2.566324e+05


---
# Parte 13 — Um mapa mental para escolher o comando

| Se eu penso… | Em SQL, procuro… |
|---|---|
| “quais colunas?” | `SELECT` |
| “de qual tabela?” | `FROM` |
| “só essas linhas” | `WHERE` |
| “quantos?” | `COUNT()` |
| “qual total?” | `SUM()` |
| “por categoria” | `GROUP BY` |
| “maiores primeiro” | `ORDER BY ... DESC` |
| “só os primeiros” | `LIMIT` |
| “preciso de informação de outra tabela” | `JOIN ... ON` |

> **Pergunta primeiro. Comando depois.**


**Mini desafios de revisão**

**Desafio A**
Quais cinco UFs possuem mais registros de licitação, independentemente do valor?

Tente montar antes de abrir a resposta.


**Desafio B**
Quais unidades gestoras aparecem em mais licitações?


**Desafio C**
Quais modalidades aparecem nas licitações do RJ e quantos registros existem de cada uma?


**Desafio A **


*"Quais cinco UFs possuem mais registros de licitação, independentemente do valor?"*



In [30]:
sql("""
SELECT
    uf,
    COUNT(*) AS total_licitacoes
FROM licitacoes
GROUP BY uf
ORDER BY total_licitacoes DESC
LIMIT 5;
""")

,uf,total_licitacoes
0,RJ,2977
1,DF,1686
2,MG,1305
3,SP,1203
4,RS,1170


**Desafio B**

*"Quais unidades gestoras aparecem em mais licitações?"*

In [35]:
sql("""
SELECT
    codigo_ug,
    COUNT(*) AS total_licitacoes
FROM licitacoes
GROUP BY codigo_ug
ORDER BY total_licitacoes DESC
LIMIT 5;
""")

,codigo_ug,total_licitacoes
0,153173,168
1,153063,156
2,113202,139
3,150232,121
4,240120,115


**Desafio C**

*"Quais modalidades aparecem nas licitações do RJ e quantos registros existem de cada uma?"*

In [36]:
sql("""
SELECT
    codigo_modalidade,
    COUNT(*) AS total_licitacoes
FROM licitacoes
WHERE uf = 'RJ'
GROUP BY codigo_modalidade
ORDER BY total_licitacoes DESC;
""")

,codigo_modalidade,total_licitacoes
0,6,1987
1,-99,336
2,7,316
3,5,292
4,3,22
5,2,22
6,1,2
